### **Student Information**
GitHub ID: FarRail

Kaggle name: Farrel G. M.

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

## 1. Model Development (10 pts Required)

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

I matched the tweet text in final_posts.json with the labels in emotion.csv using post_id. I used data_identification.csv to identify the training posts and sampleSubmission.csv to preserve the required test order. I converted the emotion labels into integer class IDs.

I created a stratified split with approximately 80% of the labeled data for training and 20% for validation, using seed 42. This kept examples of each emotion in both groups.

I kept the original tweet text, including punctuation and hashtags. For the fine-tuned MiniLM component, I tokenized the text, truncated sequences to 80 tokens, and padded each batch to its longest sequence.

For a separate clause-based component, I split posts at punctuation and words such as "but" and "however". For posts with many clauses, I retained the first three and last three clauses. This allowed the model to examine local expressions of emotion within longer posts.

### 1.2 Feature Engineering Steps

My final model used several types of features.

First, I generated normalized sentence embeddings with sentence-transformers/all-MiniLM-L6-v2. I also fine-tuned a separate MiniLM encoder and used attention-mask mean pooling to convert its token vectors into one vector per post.

Second, I created word unigram and bigram TF-IDF features, plus character n-grams of length 3 to 5. Both vectorizers used min_df=2 and sublinear term frequency. Their vocabularies and IDF weights were fitted on the training portion of each experiment.

Third, I created 22 binary features from patterns for emotion words, insults, threats, punctuation, uppercase words, the [NAME] placeholder, and first-person pronouns. I normalized these features before combining them with the sentence and TF-IDF similarities.

Finally, I generated normalized MiniLM embeddings for individual clauses. These provided a second view of the text that could retain emotional expressions within a longer post.

### 1.3 Explanation of Your Model

My final model (V12) used an ensemble approach by combining three different models to classify tweets into six emotion categories.

**1. Fine-tuned MiniLM**

I fine-tuned a pretrained MiniLM encoder with a linear classification layer to adapt its understanding of text to the competition dataset. The model used the AdamW optimizer, a batch size of 32, and six training epochs. The learning rate was set to 0.00003 for the encoder and 0.0005 for the classification layer, with both rates halved at epoch 5.

I used weighted cross-entropy loss to account for class imbalance and averaged the prediction probabilities from epochs 4 to 6 to reduce dependence on a single training epoch.

**2. Combined-feature SVM**

The second model used an SVM with a precomputed kernel that combined several types of similarity:

- MiniLM sentence embedding similarity (weight: 1.00)
- Emotion-related feature similarity (weight: 0.40)
- Word-level TF-IDF similarity (weight: 0.55)
- Character-level TF-IDF similarity (weight: 0.55)

The SVM used balanced class weights and a regularization parameter of C=1. Its prediction probabilities were combined equally with those of the fine-tuned MiniLM model to form the full-post predictions.

**3. Clause-level SVM**

The third model used an RBF SVM with C=3 to classify individual clauses within each tweet. The maximum and mean prediction probabilities across the clauses were combined to produce a prediction for the entire tweet.

The purpose of this approach was to capture specific expressions of emotion that might be overlooked when processing a tweet as a whole.

**4. Ensemble Weight Selection**

I tested four different weights for combining the full-post and clause-level predictions.

| Full-post weight | Clause-level weight | Validation macro F1 |
|---:|---:|---:|
| 100% | 0% | 0.5585 |
| **85%** | **15%** | **0.5639** |
| 70% | 30% | 0.5592 |
| 55% | 45% | 0.5586 |

The clause-level weight of 15% was selected because it achieved the highest validation macro F1 of 0.5639. A five-fold cross-fitted evaluation of the weight selection process produced a macro F1 of 0.5539.

The final contributions of each model were:

| Model | Contribution |
|---|---:|
| Fine-tuned MiniLM | 42.5% |
| Combined-feature SVM | 42.5% |
| Clause-level SVM | 15.0% |

After selecting these weights, I trained the models on all labeled training data and generated the final test predictions.

**5. Experimental Results**

| Version | Public Kaggle score |
|---|---:|
| V1 | 0.5482 |
| V2 | 0.5529 |
| V3 | 0.5504 |
| V4 | 0.5928 |
| V5 | 0.6106 |
| V6 | 0.6060 |
| V7 | 0.6121 |
| V10 | 0.6112 |
| **V12** | **0.6123** |

V12 was selected as my final submission and uploaded under the filename lab2_submission_v9.csv.

- Final public score: **0.6123**
- Final private score: **0.6135**
- Private leaderboard rank: **75**

From the results, I observed that additional training and changes to the model did not always guarantee better performance. Although the score improved considerably from V1 to V12, the later improvements became much smaller.

I also experimented with DeBERTa as an alternative encoder, which achieved a mean validation macro F1 of 0.5508 across three folds. I ultimately retained V12 because it achieved the highest recorded public and private leaderboard scores among my submissions.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used ChatGPT to help write model code, generate submission files, suggest improvements, and diagnose training errors. I supplied the competition files, reported submission scores, and shared validation output and error messages.

I used the suggested MiniLM fine-tuning and ensemble methods during development. The final v12 code combined fine-tuned MiniLM probabilities with SVM predictions based on sentence embeddings, TF-IDF features, emotion cues, and clause embeddings. I retained v12 as my selected submission because it had the highest recorded public score.

I also followed suggestions to test DeBERTa. This required fixing GPU setup and training problems, including predictions collapsing to one class and a non-finite loss. These experiments did not become my final selected submission.

The execution-related prompts are recorded below. Signup, initial file-access questions, navigation, upload instructions, and export requests are omitted. Long diagnostic output is shortened, while the requests and reported results are retained.

**Initial implementation and submission repair**

- “these are the data given, I have no idea how to do anything yet. Please help tell me what to do in the fastest and most effective way possible”
- “it says ID column id not found in submission”

**Model improvement requests and score feedback**

- “"Welcome to the leaderboard! Your submission scored 0.5482." is what they said, is there any way to get better?”
- “it gave a 0.5529, try to find a method to make it pass 0.6”
- “score given was 0.5504”
- “0.5928 was the new score, continue to train again”
- “it has changed to 0.6106, next benchmark is 0.64”
- “the new score is 0.6060, continue training to go pass the benbchmark”
- “the new score is now 0.6121”
- “continue to train until benchmark is passed”
- “this one got 0.6112, continue training”
- “this one's result is 0.6123, what needs to be done to improve substantially?”
- “continue training, this is my last submission”
- “see what else can be adjusted and continue training”
- “how come all this training leads to worse scores?”

**Training continuation and monitoring**

I also sent the following short execution requests during the experiments.

- “please continue training”
- “please inform me when its done”
- “okay continue training”
- “please continue”
- “status”
- “sorry to bother, please continue until training is finished and let me know about the progress”
- “continue training” — repeated four times in the exported record

**DeBERTa setup and debugging**

- “how to do this” — referring to the suggestion to test a stronger encoder on fresh validation splits
- “how to get/do this?”
- “got this error” — followed by the GPU availability assertion error
- “i did step 2 and got: [three training and validation fold sizes] what now?”
- “i got these values: DeBERTa fold scores: [0.0516, 0.0581, 0.0573] Mean macro F1: 0.0557 what else is needed?”
- “got this: [actual class counts] Predicted: Counter({'anger': 1124})”
- “got RuntimeError: Non-finite loss at step 2: nan”
- “give me the full code to paste and which cell to change”
- “so change def train_fold and rerun?”
- “the 24 example test got back this result: [training reached 24/24 correct] what then?”
- “got this: [training losses, class counts, and fold 1 validation macro F1 of 0.5448]”
- “git me the entire new code. Is it all in the new cell?”
- “i have 7 minuts, will there be enough time?”
- “[fold scores of 0.5448, 0.5722, and 0.5355, with mean macro F1 of 0.5508]”
- “try to train a new one asap, just try sisnce the previous try wont go away”

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
### Add the code related to the preprocessing steps in cells inside this section
from pathlib import Path
import numpy as np
import pandas as pd

# Load the original competition data
DATA_DIR = Path("competition_data")

posts = pd.read_json(DATA_DIR / "final_posts.json").set_index("post_id")
split = pd.read_csv(DATA_DIR / "data_identification.csv")
labels = pd.read_csv(DATA_DIR / "emotion.csv").set_index("post_id")
sample = pd.read_csv(DATA_DIR / "sampleSubmission.csv")

# Identify training and test posts
train_post_ids = split.loc[split["set"] == "train", "post_id"].tolist()
test_post_ids = sample.iloc[:, 0].tolist()

# Combine text, identifiers, and emotion labels
data = posts.loc[train_post_ids + test_post_ids].copy()
data["id"] = data.index
data["label"] = labels["label"].reindex(data.index)

# Convert emotion labels into integer class IDs
mask = data["label"].notna()
classes, y = np.unique(data.loc[mask, "label"], return_inverse=True)

# Create the original V12 stratified validation split
def partition(labels):
    seed = 42
    train, valid = [], []

    for label in sorted(set(labels)):
        group = list(np.flatnonzero(labels == label))

        for i in range(len(group) - 1, 0, -1):
            seed = (1664525 * seed + 1013904223) % 2**32
            j = seed * (i + 1) // 2**32
            group[i], group[j] = group[j], group[i]

        cut = int(0.2 * len(group) + 0.5)
        valid.extend(group[:cut])
        train.extend(group[cut:])

    return np.array(train), np.array(valid)

train, valid = partition(y)
test = np.flatnonzero(~mask)

print("Training examples:", len(train))
print("Validation examples:", len(valid))
print("Test examples:", len(test))
print("Emotion classes:", classes)

Training examples: 2687
Validation examples: 673
Test examples: 1440
Emotion classes: ['anger' 'disgust' 'fear' 'joy' 'sadness' 'surprise']


## 2. Feature Engineering Steps

In this section, I prepared the different types of features used in the V12 ensemble model, including MiniLM embeddings, emotion-related features, word and character TF-IDF, and clause-level embeddings. These features capture different aspects of the tweets such as from individual words to their overall meaning.

### 2.1 Tokenization and sentence embeddings

In [4]:
import re
import torch
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

MODEL = "sentence-transformers/all-MiniLM-L6-v2"

torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tokenize tweets for MiniLM fine-tuning
tokenizer = AutoTokenizer.from_pretrained(MODEL)
encoded = tokenizer(
    data.text.tolist(),
    truncation=True,
    max_length=80,
    padding=False
)

# Create batches with dynamic padding
def batches(ids):
    for start in range(0, len(ids), 32):
        batch = ids[start:start + 32]
        inputs = tokenizer.pad(
            [{key: encoded[key][i] for key in encoded} for i in batch],
            return_tensors="pt"
        )
        yield batch, inputs.to(device)

# Generate normalized embeddings for whole tweets
encoder = SentenceTransformer(MODEL, device="cpu")

vectors = encoder.encode(
    data.text.tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Sentence embedding shape:", vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence embedding shape: (4800, 384)


### 2.2 Emotion-related features

In [5]:
patterns = [re.compile(p) for p in [
    r"(?i)angry|anger|furious|rage|outrage|pissed|infuriat|irritat|annoy|frustrat|mad\b|temper",
    r"(?i)idiot|stupid|moron|dumbass|asshole|bitch|bastard|shut up|fuck off|fuck you|piece of shit|jerk\b",
    r"(?i)attack|fight|fought|protest|riot|violence|violent|blame|threaten|damn|hate|enemy",
    r"(?i)disgust|gross|nasty|repuls|revolt|nause|sicken|vomit|puke|stink|stench|filth|rotten|eww|yuck|vile|repugnant|loathe|distast",
    r"(?i)creep|ugly|hideous|horrid|worst|worse|awful|horrible|pathetic|terrible|trash|garbage|unclean|dirty",
    r"(?i)afraid|scare|scary|fright|fear|terrifi|terrify|terror|petrif|horror|horrif|panic|dread|phobia|nightmare",
    r"(?i)anxi|nervous|worri|worry|concern|unsafe|danger|unsettled|uneasy|risk|threat|menace|alarm|stress",
    r"(?i)weapon|missile|bomb|shoot|shot|soldier|gun\b|security|escape|scream|flee|fled|trembl",
    r"(?i)sad\b|sadly|sadness|depress|heartbr|sorrow|grie[fv]|mourn|devastat|unhapp|lonel|miserabl|hopeless|despair|regret|disappoint",
    r"(?i)cry\b|cri[ye]|tears|weep|sorry|miss you|miss him|miss her|unfortunat|painful|suffer|sympath|condolence|tragic|traged",
    r"(?i)death|dead|dying|died|passed away|funeral|lost|loss|illness|cancer|suicide",
    r"(?i)joy|happ|glad|delight|cheer|pleased|ecsta|elat|thrill|excite|celebrat|smil|grin|bliss",
    r"(?i)love|lovely|adore|beautiful|wonderful|fantastic|amazing|awesome|great|excellent|brilliant|perfect|favorite|favourite|best|nice|cool",
    r"(?i)laugh|funny|hilarious|haha|hehe|lol|lmao|rofl|congrat|thank|grateful|enjoy|proud",
    r"(?i)surpris|shock|astonish|amaze|stun|unexpect|unbeliev|incredib|bewilder|baffl|speechless|startl|mind.blown",
    r"(?i)wow|whoa|woah|omg|oh my god|no way|really\?|what\?|what the|never knew|didn.t know|had no idea|can.t believe|how come|turns out|apparently",
    r"\?",
    r"!",
    r"[?!]{2,}",
    r"\b[A-Z]{3,}\b",
    r"\[NAME\]",
    r"(?i)\b(I|my|me)\b"
]]

# Generate binary emotion features
cues = np.array([
    [float(bool(pattern.search(text))) for pattern in patterns]
    for text in data.text
])

# Normalize the feature vectors
cues /= np.maximum(
    np.linalg.norm(cues, axis=1, keepdims=True),
    1
)

print("Emotion feature shape:", cues.shape)

Emotion feature shape: (4800, 22)


### 2.3 TF-IDF and Combined Similarity Kernel

In [7]:
train_text = data.loc[mask, "text"].tolist()

def kernel(fit_ids, query_ids):

    # MiniLM sentence similarity
    result = vectors[query_ids] @ vectors[fit_ids].T

    # Emotion feature similarity
    result += 0.40 * (cues[query_ids] @ cues[fit_ids].T)

    # Word and character TF-IDF similarities
    for analyzer, bounds in [
        ("word", (1, 2)),
        ("char", (3, 5))
    ]:
        tf = TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=bounds,
            min_df=2,
            sublinear_tf=True
        )

        base = tf.fit_transform(
            [train_text[i] for i in fit_ids]
        )

        query_features = tf.transform(data.text.iloc[query_ids])

        result += 0.55 * (query_features @ base.T).toarray()

    return result

### 2.4 Clause-Level Embeddings

In [8]:
clauses = []
owner = []

# Split tweets into smaller clauses
for row, text in enumerate(data.text):
    pieces = [
        v.strip()
        for v in re.split(
            r"[.!?;\n]+|,\s+|\bbut\b|\bhowever\b",
            text,
            flags=re.I
        )
        if v.strip()
    ]

    # Retain the first three and up to three final clauses
    selected = pieces[:3] + pieces[3:][-3:] if pieces else [text]

    clauses.extend(selected)
    owner.extend([row] * len(selected))

# Generate embeddings for individual clauses
clause_vec = encoder.encode(
    clauses,
    normalize_embeddings=True,
    show_progress_bar=False
)

owner = np.asarray(owner)

print("Number of clauses:", len(clauses))
print("Clause embedding shape:", clause_vec.shape)

Number of clauses: 9341
Clause embedding shape: (9341, 384)


## 3. Model Implementation Steps

In this section, I trained the three components of the V12 ensemble, evaluated different combinations of their prediction probabilities, and selected the best combination using validation macro F1. The selected model is then trained on all labeled data to generate the final submission.

### 3.1 Fin-Tuned MiniLM

In [9]:
import hashlib
import inspect
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

def fit(ids, queries, seed=42):

    # Identify checkpoints based on the data and notebook code
    source = MODEL + inspect.getsource(fit) + inspect.getsource(batches)

    fingerprint = hashlib.sha256(
        pd.util.hash_pandas_object(
            data[["id", "text", "label"]], index=False
        ).values.tobytes()
        + ids.tobytes()
        + queries.tobytes()
        + source.encode("utf-8")
    ).hexdigest()[:16]

    folder = Path("v12_notebook_checkpoints")
    folder.mkdir(exist_ok=True)

    checkpoint = folder / f"{fingerprint}-{seed}.pt"
    finished = checkpoint.with_suffix(".npy")

    if finished.exists():
        return np.load(finished)

    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    # Initialize MiniLM and the classification layer
    encoder = AutoModel.from_pretrained(MODEL).to(device)
    head = torch.nn.Linear(
        encoder.config.hidden_size, len(classes)
    ).to(device)

    optimizer = torch.optim.AdamW([
        {"params": encoder.parameters(), "lr": 3e-5},
        {"params": head.parameters(), "lr": 5e-4}
    ])

    # Give more weight to less frequent emotion classes
    weight = (
        len(ids) / (len(classes) * np.bincount(y[ids]))
    ) ** 0.5

    loss_fn = torch.nn.CrossEntropyLoss(
        weight=torch.tensor(
            weight, dtype=torch.float32, device=device
        )
    )

    # Mean pooling excludes padding tokens
    def logits(inputs):
        hidden = encoder(**inputs).last_hidden_state
        attention = inputs.attention_mask.unsqueeze(-1)

        pooled = (
            (hidden * attention).sum(1) / attention.sum(1)
        )

        return head(pooled)

    history = {}
    start = 1

    # Resume training if a checkpoint exists
    if checkpoint.exists():
        state = torch.load(
            checkpoint, map_location="cpu", weights_only=True
        )

        encoder.load_state_dict(state["encoder"])
        head.load_state_dict(state["head"])
        optimizer.load_state_dict(state["optimizer"])

        rng.bit_generator.state = state["rng"]
        torch.set_rng_state(state["torch_rng"])

        if device.type == "cuda":
            torch.cuda.set_rng_state_all(state["cuda_rng"])

        history = {
            k: v.numpy() for k, v in state["history"].items()
        }

        start = state["epoch"] + 1
        del state

    # Train for six epochs
    for epoch in range(start, 7):

        # Reduce learning rates at epoch 5
        if epoch == 5:
            for group in optimizer.param_groups:
                group["lr"] *= 0.5

        encoder.train()
        order = rng.permutation(ids)

        if seed != 42:
            order = np.concatenate([
                sorted(
                    order[i:i + 192],
                    key=lambda j: len(encoded["input_ids"][j])
                )
                for i in range(0, len(order), 192)
            ])

        for indices, inputs in batches(order):
            optimizer.zero_grad()

            loss = loss_fn(
                logits(inputs),
                torch.tensor(
                    y[indices],
                    device=device,
                    dtype=torch.long
                )
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                encoder.parameters(), 1.0
            )

            optimizer.step()

        # Store predictions from epochs 4 to 6
        if epoch >= 4:
            encoder.eval()

            with torch.no_grad():
                history[epoch] = np.concatenate([
                    logits(x).softmax(1).cpu().numpy()
                    for _, x in batches(queries)
                ])

        print("Seed", seed, "completed epoch", epoch)

        # Save training progress
        tmp = checkpoint.with_suffix(".tmp")

        torch.save({
            "epoch": epoch,
            "encoder": encoder.state_dict(),
            "head": head.state_dict(),
            "optimizer": optimizer.state_dict(),
            "rng": rng.bit_generator.state,
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": (
                torch.cuda.get_rng_state_all()
                if device.type == "cuda" else []
            ),
            "history": {
                k: torch.from_numpy(v)
                for k, v in history.items()
            }
        }, tmp)

        tmp.replace(checkpoint)

    # Average probabilities across epochs 4 to 6
    result = np.mean(list(history.values()), axis=0)

    with finished.with_suffix(".tmp").open("wb") as handle:
        np.save(handle, result)

    finished.with_suffix(".tmp").replace(finished)
    checkpoint.unlink()

    return result

### 3.2 Combined-Feature SVM and Clause-Level SVM

In [10]:
# Train an SVM using the combined similarities from Section 2
def companion(ids, queries):

    model = SVC(
        C=1,
        kernel="precomputed",
        class_weight="balanced",
        probability=True,
        random_state=42
    )

    model.fit(kernel(ids, ids), y[ids])

    return model.predict_proba(kernel(ids, queries))


def models(ids, queries):

    # Combine predictions from the two full-post models
    minilm_predictions = fit(ids, queries, seed=42)

    full_post_predictions = (
        minilm_predictions + companion(ids, queries)
    ) / 2

    # Select clauses belonging to training tweets
    training = np.flatnonzero(np.isin(owner, ids))
    count = np.bincount(owner, minlength=len(data))

    # Train the clause-level SVM
    clf = SVC(
        C=3,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=42
    )

    clf.fit(
        clause_vec[training],
        y[owner[training]],
        sample_weight=1 / np.sqrt(count[owner[training]])
    )

    # Predict emotions for individual query clauses
    query_clauses = np.flatnonzero(np.isin(owner, queries))
    probs = clf.predict_proba(clause_vec[query_clauses])
    groups = owner[query_clauses]

    # Combine clause predictions into one prediction per tweet
    aggregate = []

    for row in queries:
        scores = probs[groups == row]

        combined = (
            scores.max(axis=0) + scores.mean(axis=0)
        ) / 2

        aggregate.append(combined / combined.sum())

    return full_post_predictions, np.array(aggregate)

### 3.3 Ensemble Weight Selection and Validation

In [11]:
# Generate validation predictions from the model components
base, clauses_pred = models(train, valid)

# Candidate weights for clause-level predictions
weights = [0, 0.15, 0.30, 0.45]


def predictions(a, b, w):
    return ((1 - w) * a + w * b).argmax(axis=1)


def score(rows, w):
    return f1_score(
        y[valid][rows],
        predictions(base[rows], clauses_pred[rows], w),
        average="macro"
    )


def choose(rows):
    return max(weights, key=lambda w: score(rows, w))


# Cross-fitted evaluation of weight selection
rows = np.arange(len(valid))
held = np.empty(len(rows), dtype=int)

folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

for tr, va in folds.split(base, y[valid]):
    fold_weight = choose(tr)

    held[va] = predictions(
        base[va], clauses_pred[va], fold_weight
    )

print(
    "Cross-fitted F1:",
    f1_score(y[valid], held, average="macro")
)

# Compare validation results
for w in weights:
    print("Weight:", w, "| Validation F1:", score(rows, w))

selected_weight = choose(rows)

print("Selected clause weight:", selected_weight)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Seed 42 completed epoch 1
Seed 42 completed epoch 2
Seed 42 completed epoch 3
Seed 42 completed epoch 4
Seed 42 completed epoch 5
Seed 42 completed epoch 6


c:\Users\Farrel\Documents\codes\data_mining\DM2026Labs\lab2\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Farrel\Documents\codes\data_mining\DM2026Labs\lab2\DM2026-Lab2-Exercise\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Cross-fitted F1: 0.5539434376877858
Weight: 0 | Validation F1: 0.5585056272682825
Weight: 0.15 | Validation F1: 0.563921450964011
Weight: 0.3 | Validation F1: 0.5592183902379043
Weight: 0.45 | Validation F1: 0.5585784423266723
Selected clause weight: 0.15
